In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'SAS': ['Emanuel Miller'], 'DEN': ['Zeke Nnaji']}

Out Players:
{'ATL': ['Jock Landale'], 'DET': ['Duncan Robinson', 'Caris LeVert', 'Isaiah Stewart', 'Cade Cunningham', 'Tobias Harris'], 'ORL': ['Franz Wagner', 'Jett Howard', 'Jonathan Isaac'], 'PHI': ['Cameron Payne', 'Johni Broome'], 'CLE': ['Dean Wade', 'Max Strus', 'Donovan Mitchell', 'Jaylon Tyson', 'James Harden', 'Thomas Bryant'], 'MEM': ['Jahmai Mashack', 'Ty Jerome', 'Javon Small'], 'POR': ['Shaedon Sharpe', 'Jerami Grant', 'Vít Krejčí'], 'DEN': ['Peyton Watson', 'Spencer Jones']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 10 teams with confirmed lineups
Updated 2 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
130,NaN,2025-26,1631172,Ousmane Dieng,Ousmane,1610612749,MIL,Milwaukee Bucks,22501126,2026-04-03T00:00:00,MIL vs. BOS,L,23.983333,4,15,0.267,1,4,0.250,0,0,0.0,0,3,3,1,1,0,0,1,2,0,9,-32,13.1,0,0,14.0,1,23:59,1,94.0,97.9,97.9,155.5,159.2,159.2,-61.5,-61.3,-61.3,0.091,1.0,5.9,0.000,0.158,0.060,5.9,5.9,0.300,0.300,0.296,0.297,99.19,96.07,80.06,96.07,-0.009,47,4.0,15.0,35,83,0.422,21,47,0.447,10,13,0.769,8,22,30,24,13.0,5,4,4,21,13,101,-32.0,107.8,109.8,138.0,141.5,-30.3,-31.7,0.686,1.85,19.0,0.200,0.600,0.378,0.141,0.548,0.569,95.0,93.0,77.50,92,0.322,1610612738,BOS,Boston Celtics,50,89,0.562,17,37,0.459,16,19,0.842,10,38,48,33,9.0,7,4,4,13,21,133,32.0,138.0,141.5,107.8,109.8,30.3,31.7,0.660,3.67,23.1,0.400,0.800,0.622,0.096,0.657,0.683,95.0,93.0,77.50,94,0.678,G,C,22.0
129,NaN,2025-26,1629013,Landry Shamet,Landry,1610612752,NYK,New York Knicks,22501123,2026-04-03T00:00:00,NYK vs. CHI,W,12.683333,3,7,0.429,2,4,0.500,0,0,0.0,1,0,1,1,0,1,0,0,0,0,8,12,13.7,0,0,14.0,1,12:41,1,143.6,137.0,137.0,95.9,92.6,92.6,47.8,44.4,44.4,0.083,0.0,12.5,0.083,0.000,0.037,0.0,0.0,0.571,0.571,0.233,0.243,98.09,102.18,85.15,102.18,0.105,27,3.0,7.0,48,91,0.527,15,39,0.385,25,28,0.893,13,41,54,30,9.0,11,3,2,21,23,136,40.0,136.9,136.0,96.5,96.0,40.4,40.0,0.625,3.33,20.7,0.333,0.788,0.577,0.090,0.610,0.658,99.4,100.0,83.33,100,0.713,1610612741,CHI,Chicago Bulls,35,81,0.432,11,35,0.314,15,26,0.577,9,27,36,24,16.0,4,2,3,23,21,96,-40.0,96.5,96.0,136.9,136.0,-40.4,-40.0,0.686,1.50,17.8,0.212,0.667,0.423,0.160,0.500,0.519,99.4,100.0,83.33,100,0.287,NaN,SG,28.0
128,NaN,2025-26,1630182,Josh Green,Josh,1610612766,CHA,Charlotte Hornets,22501120,2026-04-03T00:00:00,CHA vs. IND,W,24.400000,2,5,0.400,2,4,0.500,0,0,0.0,1,1,2,2,0,1,0,0,2,0,6,8,14.4,0,0,14.0,1,24:24,1,124.2,125.5,125.5,109.4,107.7,107.7,14.8,17.8,17.8,0.100,0.0,28.6,0.036,0.036,0.036,0.0,0.0,0.600,0.600,0.088,0.088,101.04,101.31,84.43,101.31,0.044,51,2.0,5.0,46,96,0.479,24,49,0.490,13,15,0.867,12,36,48,31,10.0,8,7,1,18,18,129,21.0,128.2,130.3,109.8,109.1,18.5,21.2,0.674,3.10,21.5,0.275,0.722,0.505,0.101,0.604,0.629,99.5,99.0,82.50,99,0.574,1610612754,IND,Indiana Pacers,43,93,0.462,15,36,0.417,7,10,0.700,11,34,45,29,12.0,5,1,7,18,18,108,-21.0,109.8,109.1,128.2,130.3,-18.5,-21.2,0.674,2.42,20.6,0.278,0.725,0.495,0.121,0.543,0.554,99.5,99.0,82.50,99,0.426,NaN,SG,25.0
139,NaN,2025-26,203937,Kyle Anderson,Kyle,1610612750,MIN

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260406_002052.json


,home_team,away_team,commence_time,bookmakers
0,Atlanta Hawks,New York Knicks,2026-04-06 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Orlando Magic,Detroit Pistons,2026-04-06 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Memphis Grizzlies,Cleveland Cavaliers,2026-04-07 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,San Antonio Spurs,Philadelphia 76ers,2026-04-07 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Denver Nuggets,Portland Trail Blazers,2026-04-07 01:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
BOOKMAKER = 'Underdog'
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-06 00:20:52
US latest pull: 2026-04-06 00:19:59


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
250,Underdog,player_points,Karl-Anthony Towns,Over,18.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
251,Underdog,player_points,Karl-Anthony Towns,Under,18.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
252,Underdog,player_points,Jalen Brunson,Over,24.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
253,Underdog,player_points,Jalen Brunson,Under,24.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
254,Underdog,player_points,Jonathan Kuminga,Over,10.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Karl-Anthony Towns: Response ended prematurely
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] G.G. Jackson: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90
0,Jalen Johnson,AST,28.62,34.20,40.26,0.0705,0.1685,0.2632,2.02,5.76,10.60
1,Nickeil Alexander-Walker,AST,20.10,29.08,36.65,0.0385,0.1110,0.1910,0.77,3.23,7.00
2,Desmond Bane,AST,27.43,32.47,38.75,0.0500,0.1264,0.2330,1.37,4.10,9.03
3,Evan Mobley,AST,23.12,30.32,37.68,0.0563,0.1098,0.2064,1.30,3.33,7.78
4,Cedric Coward,AST,17.94,26.05,32.16,0.0338,0.1130,0.2023,0.61,2.94,6.51
5,De'Aaron Fox,AST,22.09,30.60,38.75,0.1122,0.1790,0.3267,2.48,5.48,12.66
6,VJ Edgecombe,AST,26.51,33.68,39.33,0.0415,0.1277,0.2291,1.10,4.30,9.01


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
17,Julian Champagnie,REB,5.5,25.40,4.65,0.506,0.494
32,Jalen Suggs,PTS,14.5,27.95,13.00,0.539,0.461
23,OG Anunoby,PTS,15.5,34.49,17.70,0.670,0.330
29,Franz Wagner,PTS,11.5,29.94,21.90,0.914,0.086
50,Quentin Grimes,PTS,8.5,26.53,10.84,0.848,0.152
24,Josh Hart,PTS,12.5,30.70,10.01,0.381,0.619
43,De'Aaron Fox,PTS,15.5,30.60,20.11,0.802,0.198
47,Keldon Johnson,PTS,12.5,26.70,17.05,0.779,0.221
5,De'Aaron Fox,AST,5.5,30.60,5.48,0.626,0.374
11,James Harden,REB,4.5,33.22,4.56,0.620,0.380


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker=BOOKMAKER,
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

all_line_probs = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
35,Daniss Jenkins,PTS,17.5,28.98,13.23,0.390,0.610,PTS,Orlando Magic,-3.0,225.0,114.0,15.0,100.38,15.0,-118.0,-110.0,0.541,0.524,18.9,19.0,6.24,1.4,1.5,-0.224,0.589,0.411,8.82,-21.54,0.6,0.6,0.40,0.18,34.69,6.58,0.23,0.03,0.67,3.0
9,Ausar Thompson,REB,5.5,26.21,6.60,0.698,0.302,REB,Orlando Magic,-3.0,225.0,114.0,15.0,100.38,15.0,-118.0,-110.0,0.541,0.524,6.1,6.5,2.08,0.6,1.0,-0.288,0.613,0.387,13.25,-26.12,0.8,0.6,0.60,0.50,28.48,4.75,0.15,0.05,7.80,5.0
42,Joel Embiid,PTS,28.5,32.54,24.29,0.420,0.580,PTS,San Antonio Spurs,8.5,236.5,110.2,3.0,100.77,12.0,-104.0,-116.0,0.510,0.537,29.4,28.0,6.28,0.9,-0.5,-0.143,0.557,0.443,9.26,-17.51,0.4,0.5,0.67,0.39,33.27,4.10,0.33,0.05,9.00,1.0
40,Cedric Coward,PTS,13.5,26.05,14.37,0.629,0.371,PTS,Cleveland Cavaliers,13.5,238.5,114.0,14.0,100.60,13.0,-112.0,-115.0,0.528,0.535,13.5,14.0,5.04,0.0,0.5,0.000,0.500,0.500,-5.36,-6.52,0.6,0.5,0.47,0.45,25.24,1.62,0.22,0.07,10.00,1.0
19,Donovan Clingan,REB,11.5,25.89,10.89,0.476,0.524,REB,Denver Nuggets,8.5,240.5,116.0,21.0,99.47,20.0,-119.0,-102.0,0.543,0.505,11.4,12.5,4.27,-0.1,1.0,0.023,0.491,0.509,-9.64,0.80,0.4,0.6,0.60,0.36,25.84,3.61,0.19,0.07,11.00,6.0


### Get top EVs

In [10]:
slate_path = build_greedy_slate(
    prob_df=all_line_probs,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 54  |  Pairs: 115  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
